Directional topological descriptor comparison

In [ ]:
import numpy as np
import pandas as pd
import gudhi as gd
from utils import LowerStarLayer, ecg_comparison
from tqdm.auto import tqdm
import wfdb

def load_mitbih_beats(record_id='100', window=200):
    record = wfdb.rdrecord(f'mitbih_data/{record_id}')
    annotation = wfdb.rdann(f'mitbih_data/{record_id}', 'atr')
    signal = record.p_signal[:, 0]
    X = []
    y = []
    for i, label in enumerate(annotation.symbol):
        if label not in ['N', 'V']:
            continue
        idx = annotation.sample[i]
        if idx - window < 0 or idx + window > len(signal):
            continue
        beat = signal[idx - window:idx + window]
        X.append(beat)
        y.append(0 if label == 'N' else 1)
    return (np.array(X), np.array(y))

def load_mitbih_all(records_id, window=200, classes=('N', 'V')):
    """
    Loads all MIT-BIH records into a single dataset.

    Returns:
        X: (n_beats, 2*window)
        y: (n_beats,)
    """
    X_all = []
    y_all = []
    for rec in tqdm(records_id):
        try:
            record = wfdb.rdrecord(f'mitbih_data/{rec}')
            ann = wfdb.rdann(f'mitbih_data/{rec}', 'atr')
            signal = record.p_signal[:, 0]
            for i, label in enumerate(ann.symbol):
                if label not in classes:
                    continue
                idx = ann.sample[i]
                if idx - window < 0 or idx + window >= len(signal):
                    continue
                beat = signal[idx - window:idx + window]
                X_all.append(beat)
                y_all.append(0 if label == classes[0] else 1)
        except Exception as e:
            print(f'Skipping record {rec}: {e}')
    return (np.array(X_all), np.array(y_all))

def balance_dataset(X, y):
    idx_n = np.where(y == 0)[0]
    idx_v = np.where(y == 1)[0]
    idx_n = np.random.choice(idx_n, size=len(idx_v), replace=False)
    idx = np.concatenate([idx_n, idx_v])
    return (X[idx], y[idx])

def build_lowerstar_layer(n_points):
    stbase = gd.SimplexTree()
    for i in range(n_points - 1):
        stbase.insert([i, i + 1], -10000000000.0)
    layer = LowerStarLayer(simplextree=stbase)
    return layer

np.random.seed(42)
records = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214']
X, y = load_mitbih_all(records_id=records)
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))
X, y = balance_dataset(X, y)
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))

n_points = 400
layer = build_lowerstar_layer(n_points)
results, timing, folds = ecg_comparison(X, y, layer, n_points, output_prefix="mitbih")

display(results)
display(timing)

c:\Users\cimagroup\Documents\Repositorios\PET\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 35/35 [00:02<00:00, 17.23it/s]


(56906, 400)
(56906,)
(9344, 400)
(9344,)


Raw + SVM: 100%|██████████| 50/50 [27:41<00:00, 33.22s/it]


{'Representation': 'Raw', 'Dimension': 400, 'Model': 'SVM', 'Accuracy Mean': np.float64(0.9080906667216608), 'Accuracy Std': np.float64(0.005571196810983159), 'Balanced Accuracy Mean': np.float64(0.9080907945814107), 'Balanced Accuracy Std': np.float64(0.005570382559377053), 'F1 Mean': np.float64(0.9083088551530125), 'F1 Std': np.float64(0.005646495218861671), 'AUC Mean': np.float64(0.9610522348563824), 'AUC Std': np.float64(0.0036673619500581125), 'Fit seconds Mean': np.float64(32.76972207600251), 'Fit seconds Std': np.float64(2.9208280553615498), 'Predict seconds Mean': np.float64(0.43909290799871087), 'Predict seconds Std': np.float64(0.05916237019373364)}


Raw + Random Forest: 100%|██████████| 50/50 [22:49<00:00, 27.38s/it]


{'Representation': 'Raw', 'Dimension': 400, 'Model': 'Random Forest', 'Accuracy Mean': np.float64(0.9856057299131668), 'Accuracy Std': np.float64(0.0019739913150921776), 'Balanced Accuracy Mean': np.float64(0.9856057094435984), 'Balanced Accuracy Std': np.float64(0.0019739819550747186), 'F1 Mean': np.float64(0.9855344734408001), 'F1 Std': np.float64(0.0020055108266510933), 'AUC Mean': np.float64(0.9988395744728953), 'AUC Std': np.float64(0.00034444276670596107), 'Fit seconds Mean': np.float64(27.259034400002566), 'Fit seconds Std': np.float64(0.8319171088268593), 'Predict seconds Mean': np.float64(0.10734476398676635), 'Predict seconds Std': np.float64(0.014370255416526949)}


... (5 outputs left out)

{'Representation': 'PE', 'Dimension': 1, 'Model': 'Random Forest', 'Accuracy Mean': np.float64(0.5958262156244737), 'Accuracy Std': np.float64(0.007965778480183636), 'Balanced Accuracy Mean': np.float64(0.5958267471286743), 'Balanced Accuracy Std': np.float64(0.007966498533932534), 'F1 Mean': np.float64(0.595756009038041), 'F1 Std': np.float64(0.00956043251477319), 'AUC Mean': np.float64(0.6669564897375583), 'AUC Std': np.float64(0.009430990890629792), 'Fit seconds Mean': np.float64(4.478818678001407), 'Fit seconds Std': np.float64(0.3409360667079468), 'Predict seconds Mean': np.float64(0.24301053000148387), 'Predict seconds Std': np.float64(0.02064323781217769)}


PE + XGBoost: 100%|██████████| 50/50 [00:07<00:00,  6.75it/s]


{'Representation': 'PE', 'Dimension': 1, 'Model': 'XGBoost', 'Accuracy Mean': np.float64(0.6936326380033523), 'Accuracy Std': np.float64(0.01015724739023874), 'Balanced Accuracy Mean': np.float64(0.6936300770648924), 'Balanced Accuracy Std': np.float64(0.010158175948042294), 'F1 Mean': np.float64(0.6911539226137329), 'F1 Std': np.float64(0.012357257357031828), 'AUC Mean': np.float64(0.7437151937774115), 'AUC Std': np.float64(0.010018145296655677), 'Fit seconds Mean': np.float64(0.135317606003955), 'Fit seconds Std': np.float64(0.014904088170849712), 'Predict seconds Mean': np.float64(0.00433019801042974), 'Predict seconds Std': np.float64(0.0012430222659692153)}


BC + SVM:  54%|█████▍    | 27/50 [46:25<40:12, 104.88s/it]  